In [ ]:
from datetime import datetime

class IncreaseSpeed:
   

    def __init__(self, current_speed: int, max_speed: int, step: int = 10):
        self.current_speed = current_speed
        self.max_speed = max_speed
        self.step = step

    def __iter__(self):
        return self

    def __next__(self, step: int = 10):
        if self.current_speed >= self.max_speed:
            raise StopIteration
        print(f"INFO: Speed increases by {self.step}")
        self.current_speed = min(self.current_speed + self.step, self.max_speed)
        return self.current_speed


class DecreaseSpeed:
 

    def __init__(self, current_speed: int, min_speed: int, step: int = 10):
        self.current_speed = current_speed
        self.min_speed = min_speed
        self.step = step

    def __iter__(self):
        return self

    def __next__(self):
        if self.current_speed <= self.min_speed:
            raise StopIteration
        print(f"INFO: Speed decreases by {self.step}")
        self.current_speed = max(self.current_speed - self.step, self.min_speed)
        return self.current_speed


class Car:
    
    _total_cars_on_road = 0

    def __init__(self, max_speed: int, current_speed: int = 0):
        self.max_speed = max_speed
        self.current_speed = current_speed

        self.state = current_speed > 0
        if self.state:
            Car._total_cars_on_road += 1

        self._increaser = IncreaseSpeed(self.current_speed, self.max_speed, step=10)
        self._decreaser = DecreaseSpeed(self.current_speed, min_speed=0, step=10)

    def accelerate(self, upper_border=None, step: int = 10):

        start_speed = self.current_speed
        was_in_parking = not self.state

        if was_in_parking:
            self.state = True
            Car._total_cars_on_road += 1

        self._increaser.current_speed = self.current_speed
        self._increaser.max_speed = self.max_speed
        self._increaser.step = step

        valid_upper = (
            isinstance(upper_border, int)
            and 0 <= upper_border <= self.max_speed
        )

        if valid_upper and upper_border >= self.current_speed:
            if was_in_parking and upper_border > step:
                self.current_speed = upper_border
            else:
                while self.current_speed < upper_border:
                    try:
                        self.current_speed = next(self._increaser)
                    except StopIteration:
                        break
                    if self.current_speed > upper_border:
                        self.current_speed = upper_border
        else:
            try:
                self.current_speed = next(self._increaser)
            except StopIteration:
                self.current_speed = self.max_speed

        print(
            f"INFO: The speed of this car has been increased from {start_speed} to {self.current_speed}"
        )

    def brake(self, lower_border=None, step: int = 10):
     

        if not self.state:
            print("INFO: The car is in parking and cannot brake")
            return

        start_speed = self.current_speed

        valid_lower = (
            isinstance(lower_border, int)
            and 0 <= lower_border <= self.max_speed
            and lower_border <= self.current_speed
        )

        if valid_lower:
            self._decreaser.current_speed = self.current_speed
            self._decreaser.min_speed = lower_border
            self._decreaser.step = step

            while self.current_speed > lower_border:
                try:
                    self.current_speed = next(self._decreaser)
                except StopIteration:
                    break
        else:
            self._decreaser.current_speed = self.current_speed
            self._decreaser.min_speed = 0
            self._decreaser.step = step
            try:
                self.current_speed = next(self._decreaser)
            except StopIteration:
                self.current_speed = 0

        print(
            f"INFO: The speed of this car has been decreased from {start_speed} to {self.current_speed}"
        )

    def parking(self):
        """Take the car off the road."""

        if not self.state:
            print("Car is already in parking")
            return

        self.brake(0)

        print("Parking the car...")
        self.state = False
        Car._total_cars_on_road -= 1

    @classmethod
    def total_cars(cls):
        return cls._total_cars_on_road

    @staticmethod
    def show_weather():
        try:
            import openmeteo_requests
        except ImportError:
            import sys
            import subprocess
            from pathlib import Path

            target_dir = str(Path.cwd() / ".hw3_deps")
            subprocess.check_call(
                [
                    sys.executable,
                    "-m",
                    "pip",
                    "install",
                    "--target",
                    target_dir,
                    "openmeteo_requests",
                ]
            )
            sys.path.insert(0, target_dir)
            import openmeteo_requests

        openmeteo = openmeteo_requests.Client()
        url = "https://api.open-meteo.com/v1/forecast"
        params = {
            "latitude": 59.9386,  
            "longitude": 30.3141,  
            "current": ["temperature_2m", "apparent_temperature", "rain", "wind_speed_10m"],
            "wind_speed_unit": "ms",
            "timezone": "Europe/Moscow",
        }

        response = openmeteo.weather_api(url, params=params)[0]
        current = response.Current()
        current_temperature_2m = current.Variables(0).Value()
        current_apparent_temperature = current.Variables(1).Value()
        current_rain = current.Variables(2).Value()
        current_wind_speed_10m = current.Variables(3).Value()

        print(f"Current temperature: {round(current_temperature_2m, 0)} C")
        print(f"Current apparent_temperature: {round(current_apparent_temperature, 0)} C")
        print(f"Current rain: {current_rain} mm")
        print(f"Current wind_speed: {round(current_wind_speed_10m, 1)} m/s")


In [ ]:
car1 = Car(100, 20) 
car2 = Car(60, 30) 
car3 = Car(100, 0)
print(f"Total cars on road: {Car.total_cars()}")


car1.accelerate(100)

car2.accelerate(50)

print("Speed of car 1:", car1.current_speed)



print("Speed of car 2:", car2.current_speed)

 
car1.brake(10)

car2.brake(0)
print("Total cars on road:", Car.total_cars())
car2.parking()
print("Total cars on road:", Car.total_cars())

car3.accelerate(80)
car3.show_weather()
print("Total cars on road:", Car.total_cars())

car2.accelerate(10) 
print("Total cars on road:", Car.total_cars())



Car.show_weather()
